This code makes a FHIR Patient search request to retrieve a patient record using an NHS number identifier. Here’s a clear, structured description of what the request is doing.

High-level description

The request performs an HTTP GET against a FHIR server’s Patient endpoint, searching for a patient whose identifier matches a given NHS number.
Authentication is handled using HTTP Basic Auth, with credentials loaded from environment variables.

Breakdown of the FHIR request
1. Endpoint being called
GET {FHIR_SERVER}/Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|9999999522


Resource type: Patient

Operation: Search

Search parameter: identifier

2. Search parameter details
identifier=https://fhir.nhs.uk/Id/nhs-number|9999999522


This follows the FHIR search syntax for identifiers:

identifier = system | value


system: https://fhir.nhs.uk/Id/nhs-number
Identifies the namespace for NHS numbers (UK-specific FHIR identifier system)

value: 9999999522
The NHS number being searched for

👉 This tells the FHIR server:

“Return any Patient resource that has an identifier with this system and value.”

In [18]:
import requests
import fhirclient.models.patient as pat
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
load_dotenv()
import os
import json

fhir_password = os.getenv("FHIR_PASSWORD")
fhir_username = os.getenv("FHIR_USERNAME")
server = os.getenv("FHIR_SERVER")

nhsNumber = os.getenv("NHS_NUMBER")
if nhsNumber == None:
    nhsNumber = '9737873858'

api_url = server + "Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|"+nhsNumber
print("Calling GET "+api_url)
response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
patientJSON = response.json()
#print(patientJSON)
print(json.dumps(patientJSON, indent=4))
patientId = None

if ((patientJSON['total']> 0) and (len(patientJSON['entry'])>0)):
    patientId = patientJSON['entry'][0]['resource']['id']
    patient = pat.Patient(patientJSON['entry'][0]['resource'])
    print("Patient Id: " + patientId)
    print("Date of Birth: "+patient.birthDate.isostring)
    for id in patient.identifier:
        print('Identifier: '+ id.value + " " + id.type.coding[0].code)



Calling GET http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https://fhir.nhs.uk/Id/nhs-number|9737873858
{
    "resourceType": "Bundle",
    "id": "7ea6ac84-62f1-4dbb-9484-abbe4b1cfae0",
    "type": "searchset",
    "timestamp": "2026-07-23T06:24:17Z",
    "total": 1,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=https%3A%2F%2Ffhir.nhs.uk%2FId%2Fnhs-number%7C9737873858"
        }
    ],
    "entry": [
        {
            "fullUrl": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient/21370",
            "resource": {
                "resourceType": "Patient",
                "birthDate": "1986-09-12",
                "gender": "male",
                "id": "21370",
                "identifier": [
                    {
                        "assigner": {
                            "identifier": {
                                "system": "https://fhir.nhs.uk/Id/ods-orga

We can also search for patients using hospitals medical record number (MRN), this is known in as [Local Patient Identifier](https://www.datadictionary.nhs.uk/attributes/local_patient_identifier.html

In FHIR, identifiers are found in the identifier section. For example the previous example a MRN

```json
{
                        "assigner": {
                            "identifier": {
                                "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                                "value": "RXR"
                            }
                        },
                        "type": {
                            "coding": [
                                {
                                    "code": "MR",
                                    "system": "http://terminology.hl7.org/CodeSystem/v2-0203"
                                }
                            ]
                        },
                        "value": "RXR3302855"
                    }
```

The `type` indicates this is a MRN, the NHS Number has type of `NH` which stands for National Health Plan identifier. The `assigner` is the ODS code of the NHS Trust the MRN belongs to, and `value` is the MRN.
A `system` can also be provided like how the NHS Number has `https://fhir.nhs.uk/Id/nhs-number` which is a URI, alternatively an OID may be provided such as NHS Scotland's CHI Number which is `urn:oid:2.16.840.1.113883.2.1.3.2.4.16.53`.

To search for a MRN we can alter the previous search to

GET {FHIR_SERVER}/Patient?identifier=RXR3302855

As this MRN includes the ODS of the NHS Trust, we should only get one result back but as we are not including a `system` we may get multiple results back and so may need to filter the response on ODS codes.

In [19]:
api_url = server + "Patient?identifier=RXR3302855"

response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
patientJSON = response.json()

print(json.dumps(patientJSON, indent=4))

{
    "resourceType": "Bundle",
    "id": "8859cd09-0736-4574-8dfc-a24294073b65",
    "type": "searchset",
    "timestamp": "2026-07-23T06:24:17Z",
    "total": 1,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=RXR3302855"
        }
    ],
    "entry": [
        {
            "fullUrl": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient/21370",
            "resource": {
                "resourceType": "Patient",
                "birthDate": "1986-09-12",
                "gender": "male",
                "id": "21370",
                "identifier": [
                    {
                        "assigner": {
                            "identifier": {
                                "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                                "value": "X24"
                            }
                        },
                        "extension": [
         

We can also search on CHI Numbers e.g.

GET {FHIR_SERVER}/Patient?identifier=urn:oid:2.16.840.1.113883.2.1.3.2.4.16.53|9111111111

> Note: Do not use this 111 111 1111 as test CHI Number or NHS Number as it is an actual patient identifier. Use numbers from the approved test range.

In [20]:
api_url = server + "Patient?identifier=urn:oid:2.16.840.1.113883.2.1.3.2.4.16.53|9111111111"

response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
patientJSON = response.json()

print(json.dumps(patientJSON, indent=4))

{
    "resourceType": "Bundle",
    "id": "1cfc9f12-536e-4596-b25c-39b60eb8a3f5",
    "type": "searchset",
    "timestamp": "2026-07-23T06:24:18Z",
    "total": 0,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/Patient?identifier=urn%3Aoid%3A2.16.840.1.113883.2.1.3.2.4.16.53%7C9111111111"
        }
    ]
}


Having found the patient, we can now use the `id` to find orders and reports for the patient.

For orders which is known as ServiceRequest in FHIR, this is:

GET {FHIR_SERVER}/ServiceRequest?id={patientId}

The `patientId` was obtained in the first example value.

In [21]:
import json
import fhirclient.models.servicerequest as sr
import fhirclient.models.meta as meta
import pandas as pd
from dateutil import parser

def issued(issued):
    if issued == None:
        return None
    return parser.parse(issued.isostring)

def lastUpdated(meta : meta.Meta):
    if meta == None:
        return None
    return parser.parse(meta.lastUpdated.isostring)

if patientId != None:
    api_url = server + "ServiceRequest?patient="+patientId
    response = requests.get(api_url, auth=HTTPBasicAuth(fhir_username, fhir_password))
    response1JSON = response.json()

    print(response1JSON['total'])

    print(json.dumps(response1JSON, indent=4))

    serviceRequests = []
    for entry in response1JSON['entry']:
        #print(entry['resource'])
        print(entry['resource']['resourceType'], entry['resource']['status'])

        order = (sr.ServiceRequest(entry['resource']))
        serviceRequests.append(order)

    dfSR = pd.DataFrame([vars(s) for s in serviceRequests])
    ##dfDR['issued'] = dfDR['issued'].apply(issued)
    #dfDR['effectiveDateTime'] = dfDR['effectiveDateTime'].apply(issued)
    #dfDR['lastUpdated'] = dfDR['meta'].apply(lastUpdated)

dfSR

3
{
    "resourceType": "Bundle",
    "id": "27ccfaad-b017-453c-b6cd-39c00b1199b8",
    "type": "searchset",
    "timestamp": "2026-07-23T06:24:18Z",
    "total": 3,
    "link": [
        {
            "relation": "self",
            "url": "http://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest?patient=21370"
        }
    ],
    "entry": [
        {
            "fullUrl": "http://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest/21376",
            "resource": {
                "resourceType": "ServiceRequest",
                "category": [
                    {
                        "coding": [
                            {
                                "code": "116148004",
                                "system": "http://snomed.info/sct"
                            }
                        ]
                    }
                ],
                "code": {
                    "coding": [
                        {
                            "code": "M4.14",
       

,asNeededBoolean,asNeededCodeableConcept,authoredOn,basedOn,bodySite,category,code,doNotPerform,encounter,identifier,...,extension,modifierExtension,text,id,implicitRules,language,meta,_server,_resolved,_owner
0,None,None,None,None,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21376,None,None,<fhirclient.models.meta.Meta object at 0x11118...,None,None,None
1,None,None,<fhirclient.models.fhirdatetime.FHIRDateTime o...,[<fhirclient.models.fhirreference.FHIRReferenc...,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21377,None,None,<fhirclient.models.meta.Meta object at 0x11133...,None,None,None
2,None,None,None,None,None,[<fhirclient.models.codeableconcept.CodeableCo...,<fhirclient.models.codeableconcept.CodeableCon...,None,None,[<fhirclient.models.identifier.Identifier obje...,...,None,None,None,21940,None,None,<fhirclient.models.meta.Meta object at 0x11133...,None,None,None
